In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pandas as pd
from scipy.interpolate import make_interp_spline
from matplotlib.path import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FixedLocator
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from shapely.geometry import MultiPolygon, Polygon
import shapely  # shapely>=2.0 needed for contains_xy
import os

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False

# ================== 输出目录 ==================
output_dir = r"E:\关中干旱小论文\研究区域图-图一"
os.makedirs(output_dir, exist_ok=True)

# ================== 基础设置 ==================
years_clim  = range(1991, 2021)
year_target = 2025

start_date = pd.to_datetime("2025-03-01")
end_date   = pd.to_datetime("2025-05-31")

# =========================================================
# 西太副高：100–180E，0–50N
# =========================================================
lon_min, lon_max = 100, 180
lat_min, lat_max = 0, 50

lon_slice = slice(lon_min, lon_max)
lat_slice_hgt = slice(lat_max, lat_min)  # HGT lat 常见 北->南（递减）

level_use = 500
contour_level = 5880

# ================== 时间分段（15天，取前6段） ==================
date_ranges = []
cur = start_date
while cur <= end_date:
    pe = min(cur + timedelta(days=14), end_date)
    date_ranges.append((cur, pe))
    cur += timedelta(days=15)
date_ranges = date_ranges[:6]

# 坐标刻度（稀疏）
lon_ticks_sparse = [100, 120, 140, 160, 180]
lat_ticks_sparse = [0, 20, 40]

# ================== SST 距平设置（优化配色，增强对比度） ==================
anom_levels = np.arange(-3.0, 3.0 + 0.5, 0.5)
ANOM_VMIN = float(anom_levels.min())
ANOM_VMAX = float(anom_levels.max())

# 🌟 优化点：扩充了紫色到橙色的过渡层级，加深两端颜色，使变化更明显
anom_colors = [
    "#2d004b", # 极冷：深紫
    "#542788", # 冷：中紫
    "#998ec3", # 微冷：浅紫
    "#ffffff", # 正常：纯白
    "#f1a340", # 微暖：浅橙
    "#e66101", # 暖：亮橙
    "#b35806"  # 极暖：深棕橙
]
anom_cmap = mcolors.LinearSegmentedColormap.from_list("Enhanced_Purple_White_Orange", anom_colors, N=256)

anom_norm = mcolors.Normalize(vmin=ANOM_VMIN, vmax=ANOM_VMAX)

sst_alpha  = 0.75  # 🌟 优化点：稍微提高透明度阈值，让颜色更实、更显眼
sst_stride = 2

# ================== 线条样式 ==================
LW_YR = 1.00
LW_CL = 0.90
RIDGE_LW_YR = 1.20
RIDGE_LW_CL = 1.10

COLOR_YR = "blue"
RIDGE_COLOR_YR = "red"

COLOR_CL = (0.10, 0.10, 0.80, 0.75)
RIDGE_COLOR_CL = (0.80, 0.10, 0.10, 0.75)

# ================== 读取与处理函数 ==================
def open_hgt_year(year: int) -> xr.DataArray:
    fp = f"E:/hgt/hgt.{year}.nc"
    da = xr.open_dataset(fp)["hgt"]
    da = da.sel(level=level_use, lon=lon_slice, lat=lat_slice_hgt)
    da = da.sel(time=da.time.dt.month.isin([3, 4, 5]))
    if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0):
        da = da.where(~((da.time.dt.month == 2) & (da.time.dt.day == 29)), drop=True)
    return da

def mean_period_by_slice(ds, start, end):
    sub = ds.sel(time=slice(start, end))
    return sub.mean("time") if sub.time.size > 0 else None

def mean_period_by_monthday(ds_all, start, end):
    t = ds_all.time
    cond = (
        ((t.dt.month == start.month) & (t.dt.day >= start.day)) |
        ((t.dt.month >  start.month) & (t.dt.month < end.month)) |
        ((t.dt.month == end.month)   & (t.dt.day <= end.day))
    )
    sub = ds_all.sel(time=cond)
    return sub.mean("time") if sub.time.size > 0 else None

def mean_sst_year_monthday(ds, start, end, year):
    t = ds.time
    cond = (t.dt.year == year) & (
        ((t.dt.month == start.month) & (t.dt.day >= start.day)) |
        ((t.dt.month >  start.month) & (t.dt.month < end.month)) |
        ((t.dt.month == end.month)   & (t.dt.day <= end.day))
    )
    sub = ds.sel(time=cond)
    return sub.mean("time") if sub.time.size > 0 else None

def smooth_ridge_line(lons, lats):
    if len(lons) < 4:
        return lons, lats
    try:
        spline = make_interp_spline(lons, lats, k=3)
        lon_smooth = np.linspace(lons.min(), lons.max(), 140)
        lat_smooth = spline(lon_smooth)
        lat_smooth = np.clip(lat_smooth, lat_min, lat_max)
        return lon_smooth, lat_smooth
    except Exception:
        return lons, lats

def close_contour_path(vertices, lon_bounds=(lon_min, lon_max), lat_bounds=(lat_min, lat_max)):
    lons, lats = vertices[:, 0], vertices[:, 1]
    closed = vertices.tolist()

    at_left   = np.any(lons <= lon_bounds[0] + 0.1)
    at_right  = np.any(lons >= lon_bounds[1] - 0.1)
    at_bottom = np.any(lats <= lat_bounds[0] + 0.1)
    at_top    = np.any(lats >= lat_bounds[1] - 0.1)

    if at_left or at_right or at_bottom or at_top:
        first, last = vertices[0], vertices[-1]
        boundary_points = []
        if at_left:
            boundary_points += [[lon_bounds[0], last[1]], [lon_bounds[0], first[1]]]
        if at_top:
            boundary_points += [[last[0], lat_bounds[1]], [first[0], lat_bounds[1]]]
        if at_right:
            boundary_points += [[lon_bounds[1], last[1]], [lon_bounds[1], first[1]]]
        if at_bottom:
            boundary_points += [[last[0], lat_bounds[0]], [first[0], lat_bounds[0]]]
        closed.extend(boundary_points)
        closed.append(closed[0])

    return np.array(closed)

def extract_ridge_lines(hgt_data, contour_paths):
    ridge_lines = []
    for path in contour_paths:
        vertices = close_contour_path(path.vertices)
        if len(vertices) < 3:
            continue

        contour_path = Path(vertices)
        lon_grid, lat_grid = np.meshgrid(hgt_data.lon.values, hgt_data.lat.values)
        points = np.vstack((lon_grid.ravel(), lat_grid.ravel())).T
        inside = contour_path.contains_points(points, radius=0.1).reshape(lon_grid.shape)

        masked = hgt_data.where(inside)
        ridge_lons, ridge_lats = [], []
        for lon in hgt_data.lon.values:
            col = masked.sel(lon=lon)
            valid = col.where(col.notnull(), drop=True)
            if valid.size > 0:
                max_idx = valid.argmax(dim="lat")
                rlat = valid.lat[max_idx].values
                if contour_path.contains_point((lon, rlat), radius=0.1):
                    ridge_lons.append(lon)
                    ridge_lats.append(rlat)

        if len(ridge_lons) > 0:
            ridge_lons, ridge_lats = smooth_ridge_line(np.array(ridge_lons), np.array(ridge_lats))
            ridge_lines.append((ridge_lons, ridge_lats))
    return ridge_lines

def build_land_mask_for_sst(sst_da):
    land_feat = cfeature.NaturalEarthFeature("physical", "land", "50m")
    geoms = []
    for geom in land_feat.geometries():
        if isinstance(geom, MultiPolygon):
            geoms.extend(geom.geoms)
        elif isinstance(geom, Polygon):
            geoms.append(geom)
    land_mp = MultiPolygon(geoms)
    lons = sst_da.lon.values
    lats = sst_da.lat.values
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    mask = shapely.contains_xy(land_mp, lon_grid, lat_grid)
    return mask

def mask_land(da2d, land_mask_2d):
    return da2d.where(~land_mask_2d)

def draw_domain_border(ax, lw=0.9, alpha=0.95):
    xs = [lon_min, lon_max, lon_max, lon_min, lon_min]
    ys = [lat_min, lat_min, lat_max, lat_max, lat_min]
    ax.plot(xs, ys, transform=ccrs.PlateCarree(), color='k', lw=lw, alpha=alpha, zorder=30)

# ================== 单个子图（引入 show_ticks 参数） ==================
def plot_panel(ax, hm_yr, hm_clim, sst_anom, land_mask_2d, title_text, show_ticks=True):
    proj = ccrs.PlateCarree()
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=proj)

    if sst_anom is not None:
        da = sst_anom.sortby("lat")
        da = mask_land(da, land_mask_2d)

        lons = da.lon.values[::sst_stride]
        lats = da.lat.values[::sst_stride]
        A2d  = da.values[::sst_stride, ::sst_stride]

        if A2d.ndim == 2 and A2d.shape[0] >= 2 and A2d.shape[1] >= 2:
            A2d = np.clip(A2d, ANOM_VMIN, ANOM_VMAX)
            ax.contourf(
                lons, lats, A2d,
                levels=anom_levels, cmap=anom_cmap, norm=anom_norm,
                alpha=sst_alpha, transform=proj
            )

    ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.35, edgecolor="k", alpha=0.9)

    # 🌟 优化点：仅在指定子图绘制刻度
    if show_ticks:
        ax.set_xticks(lon_ticks_sparse, crs=proj)
        ax.set_yticks(lat_ticks_sparse, crs=proj)
        ax.xaxis.set_major_formatter(LongitudeFormatter(number_format='.0f'))
        ax.yaxis.set_major_formatter(LatitudeFormatter(number_format='.0f'))
        ax.tick_params(labelsize=8, length=3, width=0.8, pad=2)
    else:
        # 隐藏刻度线和标签，但保留边框
        ax.set_xticks([])
        ax.set_yticks([])

    draw_domain_border(ax)

    # WPSH clim
    hc = hm_clim.sortby("lat")
    cs_cl = ax.contour(
        hc.lon.values, hc.lat.values, hc.values,
        levels=[contour_level],
        colors=[COLOR_CL],
        linewidths=LW_CL,
        linestyles="--",
        transform=proj,
        zorder=40
    )
    paths_cl = cs_cl.get_paths()
    ridges_cl = extract_ridge_lines(hc, paths_cl)
    for rlon, rlat in ridges_cl:
        ax.plot(
            rlon, rlat,
            color=RIDGE_COLOR_CL, lw=RIDGE_LW_CL, ls="--",
            transform=proj, zorder=41
        )

    # WPSH 2025
    hm = hm_yr.sortby("lat")
    cs_yr = ax.contour(
        hm.lon.values, hm.lat.values, hm.values,
        levels=[contour_level],
        colors=[COLOR_YR],
        linewidths=LW_YR,
        linestyles="-",
        transform=proj,
        zorder=50
    )
    paths_yr = cs_yr.get_paths()
    ridges_yr = extract_ridge_lines(hm, paths_yr)
    for rlon, rlat in ridges_yr:
        ax.plot(
            rlon, rlat,
            color=RIDGE_COLOR_YR, lw=RIDGE_LW_YR, ls="-",
            transform=proj, zorder=51
        )

    ax.set_title(title_text, fontsize=9, pad=4)

def finalize_layout_with_cbar(fig, sm, cbar_label, outpath):
    plt.tight_layout(rect=[0.02, 0.02, 0.90, 0.95])

    cax = fig.add_axes([0.92, 0.18, 0.018, 0.64])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(cbar_label)

    cbar.ax.yaxis.set_major_locator(FixedLocator([]))
    cbar.ax.yaxis.set_minor_locator(FixedLocator([]))

    minor_ticks = np.arange(ANOM_VMIN, ANOM_VMAX + 1e-6, 0.5)
    major_ticks = np.arange(ANOM_VMIN, ANOM_VMAX + 1e-6, 1.0)

    cbar.set_ticks(major_ticks)
    cbar.ax.yaxis.set_minor_locator(FixedLocator(minor_ticks))
    cbar.ax.tick_params(which="major", labelsize=8, length=4, width=0.8)
    cbar.ax.tick_params(which="minor", length=2, width=0.6)
    cbar.ax.set_ylim(ANOM_VMIN, ANOM_VMAX)

    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", outpath)

def plot_flat(outfile):
    fig = plt.figure(figsize=(10.8, 6.6))
    axs = [fig.add_subplot(2, 3, i + 1, projection=ccrs.PlateCarree()) for i in range(6)]

    for i, (start, end) in enumerate(date_ranges):
        print(f"Panel {i+1}/6: {start.strftime('%m-%d')} to {end.strftime('%m-%d')}")

        hm_yr   = mean_period_by_slice(hgt_2025, start, end)
        hm_clim = mean_period_by_monthday(hgt_clim_all, start, end)

        sst_yr   = mean_sst_year_monthday(sst_all, start, end, year_target)
        sst_clim = mean_period_by_monthday(sst_clim_all, start, end)
        sst_anom = (sst_yr - sst_clim) if (sst_yr is not None and sst_clim is not None) else None

        if hm_yr is None or hm_clim is None:
            print("  [skip] empty HGT window.")
            axs[i].set_title("Empty", fontsize=9)
            continue
        
        # 🌟 优化点：仅当 i == 0（第一个子图）时，传入 show_ticks=True
        is_first_panel = (i == 0)
        plot_panel(axs[i], hm_yr, hm_clim, sst_anom, land_mask_2d, f"{start:%m-%d} – {end:%m-%d}", show_ticks=is_first_panel)

    fig.suptitle(
        f"MAM SST Anomaly (2025 − 1991–2020) + WPSH {contour_level} gpm (flat panels, 6 periods)\n"
        f"WPSH: 2025 solid; Clim dashed\n"
        f"Domain: 100–180E, 0–50N",
        fontsize=11, y=0.98
    )

    sm = ScalarMappable(norm=anom_norm, cmap=anom_cmap)
    sm.set_array([])
    finalize_layout_with_cbar(fig, sm, "SST Anomaly (°C)  [2025 − 1991–2020]", outfile)

# ================== 运行 ==================
if __name__ == "__main__":
    print("Loading HGT target year...")
    hgt_2025 = open_hgt_year(year_target)

    print("Loading HGT climatology years...")
    hgt_clim_all = xr.concat([open_hgt_year(y) for y in years_clim], dim="time")

    print("Loading SST...")
    sst_path = r"E:\SST\sst.mon.mean.nc"
    sst_all = xr.open_dataset(sst_path)["sst"]
    sst_all = sst_all.sel(lon=lon_slice, lat=slice(lat_min, lat_max))
    sst_all = sst_all.sel(time=sst_all.time.dt.month.isin([3, 4, 5]))
    sst_all = sst_all.where(sst_all > -5)
    sst_clim_all = sst_all.sel(time=slice("1991-01-01", "2020-12-31"))

    print("Building land mask...")
    # ✅ 拼写已修复，正常调用
    land_mask_2d = build_land_mask_for_sst(sst_all)

    out = os.path.join(output_dir, f"wpsh_flat_schemeA_{contour_level}_sst_anom_100-180_0-50_optimized.pdf")
    plot_flat(out)

    hgt_2025.close()
    hgt_clim_all.close()
    sst_all.close()
    sst_clim_all.close()
    print("✅ 西太副高（方案A优化版）已生成。")
